## Section 1 : Imports & Helper Functions

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

In [4]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.inspection import permutation_importance

In [5]:
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

In [6]:
MODEL_COLORS = {
    'Logistic Regression': '#3498DB',
    'Decision Tree':       '#E67E22',
    'Random Forest':       '#2ECC71',
    'SVC':                 '#9B59B6'
}
SCALED_MODELS = {'Logistic Regression', 'SVC'}

In [7]:
# HELPER FUNCTIONS

def get_X(name, X_scaled, X_raw):
    return X_scaled if name in SCALED_MODELS else X_raw

In [8]:
def compute_all_metrics(models, X_scaled, X_raw, y_test):
    rows = []
    for name, model in models.items():
        X = get_X(name, X_scaled, X_raw)
        y_pred = model.predict(X)
        y_prob = model.predict_proba(X)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        rows.append({
            'Model':     name,
            'Accuracy':  accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall':    recall_score(y_test, y_pred, zero_division=0),
            'F1 Score':  f1_score(y_test, y_pred, zero_division=0),
            'ROC-AUC':   auc(fpr, tpr)
        })
    return pd.DataFrame(rows).round(4).sort_values('F1 Score', ascending=False).reset_index(drop=True)

In [10]:
def plot_confusion_matrix(y_true, y_pred, title, ax, acc, f1):
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, None]
    labels = ['No Osteoporosis', 'Osteoporosis']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels,
                linewidths=1, linecolor='white', cbar=False)
    for r in range(2):
        for c in range(2):
            ax.text(c+0.5, r+0.72, f'({cm_pct[r,c]*100:.1f}%)',
                    ha='center', va='center', fontsize=8, color='dimgray')
    ax.set_title(f'{title}\\nAcc={acc:.3f}  F1={f1:.3f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

print(' Helper functions loaded.')

 Helper functions loaded.


## Section 2 : Load Models & Test Data